In [1]:
## importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats


In [2]:
df = pd.read_excel('finalforms.xlsx')

In [3]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns',None)

In [4]:
print(df.shape)
# print(df.head())
df.columns[18]

(62, 104)


'Upon arrival, I prefer to interact with a human staff member rather than a digital system.'

In [5]:
# Clean column names: strip spaces, replace line breaks, compress spaces
df.columns = (
    df.columns
    .str.replace("\n", " ", regex=False)
    .str.replace("\r", " ", regex=False)
    .str.strip()
    .str.replace(" +", " ", regex=True)
)



In [12]:
# ----------------------------
# 0) Define IV + DVs + Likert order
# ----------------------------
iv_col = "When dining at a restaurant, what is your preferred way to place an order?"


In [7]:
# # Clean IV values
# df[iv_col] = df[iv_col].astype(str).str.strip()

# # Map Likert text to numeric
# for c in voice_cols:
#     df[c] = df[c].astype(str).str.lower().str.strip().replace(LIKERT_ORDER)
#     df[c] = pd.to_numeric(df[c], errors="coerce")


In [8]:
# df[iv_col].value_counts()


In [7]:
human_label = "Human"
digital_label = "Digital"


In [11]:
# df['When dining at a restaurant, what is your preferred way to place an order?']

In [12]:
# import numpy as np
# import pandas as pd
# from scipy.stats import mannwhitneyu
# from statsmodels.stats.multitest import multipletests

# # ----------------------------
# # 1) IV (2 groups in practice) + voice DVs
# # ----------------------------
# iv_col = "When dining at a restaurant, what is your preferred way to place an order?"

# voice_cols = [
#     "I would feel comfortable speaking to a voice-based system in a restaurant",
#     "Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming",
#     "Browsing or selecting menu items by speaking would feel easy and natural, even with voice feedback",
#     "I would trust a voice-based system to correctly understand what I say",
#     "I would ask for human help only for special requests, such as menu changes or unusual needs",
#     "I would feel confident making payments through a voice-based system",
#     "I would be willing to use a voice-based system on my next restaurant visit"
# ]

# LIKERT_ORDER = {
#     "strongly disagree": 1,
#     "disagree": 2,
#     "slightly disagree": 3,
#     "slightly agree": 4,
#     "agree": 5,
#     "strongly agree": 6,
# }

# # ----------------------------
# # 2) Column presence check
# # ----------------------------
# cols_needed = [iv_col] + voice_cols
# missing = [c for c in cols_needed if c not in df.columns]
# if missing:
#     raise KeyError(f"Missing columns:\n{missing}")

# # ----------------------------
# # 3) Clean strings
# # ----------------------------
# df_clean = df[cols_needed].copy()
# df_clean = df_clean.apply(lambda s: s.astype("string").str.strip().str.lower())

# # ----------------------------
# # 4) Normalize IV to 2 groups (human vs digital), drop voice preference if present
# # ----------------------------
# IV_MAP = {
#     "human waiter": "human",
#     "digital screen (tablet / kiosk / qr code / app)": "digital",
#     "voice-based system (robot / assistant/humanoids)": "voice"  # will be excluded
# }

# iv_norm = df_clean[iv_col].map(IV_MAP)

# # Keep only human/digital rows
# keep_mask = iv_norm.isin(["human", "digital"])
# iv_norm = iv_norm[keep_mask]

# # ----------------------------
# # 5) Map voice Likert items to ordinal ranks (ONLY voice cols)
# # ----------------------------
# df_voice_ord = df_clean.loc[keep_mask, voice_cols].apply(lambda s: s.map(LIKERT_ORDER))

# # ----------------------------
# # 6) Mann–Whitney U per voice item + FDR across 7 tests
# # ----------------------------
# results = []
# for v in voice_cols:
#     tmp = pd.DataFrame({"group": iv_norm, "y": df_voice_ord[v]}).dropna()

#     g_h = tmp.loc[tmp["group"] == "human", "y"]
#     g_d = tmp.loc[tmp["group"] == "digital", "y"]

#     n_h, n_d = len(g_h), len(g_d)

#     if n_h < 3 or n_d < 3:
#         u, p = np.nan, np.nan
#     else:
#         # two-sided by default (no directional assumption)
#         u, p = mannwhitneyu(g_h, g_d, alternative="two-sided")

#     results.append({
#         "voice_item": v,
#         "n_human": n_h,
#         "n_digital": n_d,
#         "u_stat": u,
#         "p_value": p
#     })

# res = pd.DataFrame(results)

# # FDR (Benjamini-Hochberg) across the 7 p-values
# pvals = res["p_value"].to_numpy()
# mask = np.isfinite(pvals)

# res["p_fdr_bh"] = np.nan
# if mask.sum() > 0:
#     _, qvals, _, _ = multipletests(pvals[mask], alpha=0.05, method="fdr_bh")
#     res.loc[mask, "p_fdr_bh"] = qvals

# res = res.sort_values(["p_fdr_bh", "p_value"], na_position="last").reset_index(drop=True)
# res


In [9]:
df[iv_col].value_counts(dropna=False)


NameError: name 'iv_col' is not defined

In [10]:
IV_MAP = {
    "human waiter": "human",
    "digital screen (tablet / kiosk / qr code / app)": "digital",
    "voice-based system (robot / assistant / humanoids)": "voice"
}

df["modality_pref"] = (
    df[iv_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(IV_MAP)
)


NameError: name 'iv_col' is not defined

In [11]:
df_hd = df[df["modality_pref"].isin(["human", "digital"])].copy()


KeyError: 'modality_pref'

In [ ]:
df["modality_pref"].value_counts(dropna=False)


In [20]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ----------------------------
# 0) Define IV + DVs + Likert order
# ----------------------------
iv_col = "When dining at a restaurant, what is your preferred way to place an order?"

voice_cols = [
    "I would feel comfortable speaking to a voice-based system in a restaurant",
    "Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming",
    "Browsing or selecting menu items by speaking would feel easy and natural, even with voice feedback",
    "I would trust a voice-based system to correctly understand what I say",
    "I would ask for human help only for special requests, such as menu changes or unusual needs",
    "I would feel confident making payments through a voice-based system",
    "I would be willing to use a voice-based system on my next restaurant visit"
]

LIKERT_ORDER = {
    "strongly disagree": 1,
    "disagree": 2,
    "slightly disagree": 3,
    "slightly agree": 4,
    "agree": 5,
    "strongly agree": 6,
}

IV_MAP = {
    "human waiter": "human",
    "digital screen (tablet / kiosk / qr code / app)": "digital",
    "voice-based system (robot / assistant / humanoids)": "voice"
}

# ----------------------------
# 1) Quick check: what IV values exist
# ----------------------------
print(df[iv_col].value_counts(dropna=False))

# ----------------------------
# 2) Create IV group variable: modality_pref
# ----------------------------
df = df.copy()
df["modality_pref"] = (
    df[iv_col]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(IV_MAP)
)

print("\nMapped IV (modality_pref) counts:")
print(df["modality_pref"].value_counts(dropna=False))

# Keep only Human vs Digital preferers
df_hd = df[df["modality_pref"].isin(["human", "digital"])].copy()

# ----------------------------
# 3) Convert Voice Likert text -> 1..6 (ordinal)
# ----------------------------
for c in voice_cols:
    df_hd[c] = (
        df_hd[c]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(LIKERT_ORDER)
    )
    df_hd[c] = pd.to_numeric(df_hd[c], errors="coerce")  # ensures numeric for scipy

# ----------------------------
# 4) Mann–Whitney U per voice item
# ----------------------------
results = []

for v in voice_cols:
    tmp = df_hd[["modality_pref", v]].dropna()
    g_h = tmp.loc[tmp["modality_pref"] == "human", v].to_numpy(dtype=float)
    g_d = tmp.loc[tmp["modality_pref"] == "digital", v].to_numpy(dtype=float)

    n_h, n_d = len(g_h), len(g_d)

    if n_h < 3 or n_d < 3:
        U, p = np.nan, np.nan
    else:
        U, p = mannwhitneyu(g_h, g_d, alternative="two-sided")

    results.append({
        "voice_item": v,
        "n_human": n_h,
        "n_digital": n_d,
        "U_stat": U,
        "p_value": p
    })

res = pd.DataFrame(results)

# ----------------------------
# 5) BH/FDR correction across 7 tests
# ----------------------------
mask = res["p_value"].notna()
res["p_fdr_bh"] = np.nan

if mask.sum() > 0:
    res.loc[mask, "p_fdr_bh"] = multipletests(
        res.loc[mask, "p_value"],
        alpha=0.05,
        method="fdr_bh"
    )[1]

# Sort smallest adjusted p-values first
res = res.sort_values(["p_fdr_bh", "p_value"], na_position="last").reset_index(drop=True)

res


When dining at a restaurant, what is your preferred way to place an order?
Human waiter                                       33
Digital screen (tablet / kiosk / QR code / app)    25
NaN                                                 4
Name: count, dtype: int64

Mapped IV (modality_pref) counts:
modality_pref
human      33
digital    25
NaN         4
Name: count, dtype: int64


,voice_item,n_human,n_digital,U_stat,p_value,p_fdr_bh
0,Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming,32,24,257.5,0.032705,0.192696
1,"I would ask for human help only for special requests, such as menu changes or unusual needs",32,24,279.5,0.076131,0.192696
2,I would feel comfortable speaking to a voice-based system in a restaurant,33,25,304.0,0.082584,0.192696
3,I would trust a voice-based system to correctly understand what I say,32,25,330.5,0.252312,0.441546
4,I would be willing to use a voice-based system on my next restaurant visit,33,24,353.0,0.482122,0.674970
5,I would feel confident making payments through a voice-based system,32,25,424.5,0.693421,0.716539
6,"Browsing or selecting menu items by speaking would feel easy and natural, even with voice feedback",32,25,377.5,0.716539,0.716539


In [21]:
# Ensure this exists:
# LIKERT_LABELS = {1:"strongly disagree", 2:"disagree", 3:"slightly disagree",
#                 4:"slightly agree", 5:"agree", 6:"strongly agree"}


LIKERT_LABELS = {
    1: "strongly disagree",
    2: "disagree",
    3: "slightly disagree",
    4: "slightly agree",
    5: "agree",
    6: "strongly agree",
}

res_sorted = res.sort_values(["p_fdr_bh", "p_value"], na_position="last")

for _, row in res_sorted.iterrows():
    q = row["voice_item"]
    U = row["U_stat"]
    p = row["p_value"]
    p_bh = row.get("p_fdr_bh", np.nan)

    print("\n" + "=" * 120)
    print("VOICE QUESTION:")
    print(q)
    print("-" * 120)
    print(f"U statistic: {U:.4f}" if pd.notna(U) else "U statistic: NA")
    print(f"p-value:     {p:.6f}" if pd.notna(p) else "p-value:     NA")
    print(f"p (BH-FDR):  {p_bh:.6f}" if pd.notna(p_bh) else "p (BH-FDR):  NA")
    print("-" * 120)

    counts = (
        df_hd[q]
        .value_counts(dropna=False)
        .sort_index()
        .reset_index()
    )

    counts.columns = ["response_code", "count"]
    counts["response_text"] = counts["response_code"].map(LIKERT_LABELS)

    # max count among actual response codes (exclude NaN)
    if counts["response_code"].notna().any():
        max_count = counts.loc[counts["response_code"].notna(), "count"].max()
        counts["most_frequent"] = counts["count"].apply(
            lambda x: "⭐ most frequent" if x == max_count else ""
        )
    else:
        counts["most_frequent"] = ""

    print(counts)



VOICE QUESTION:
Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming
------------------------------------------------------------------------------------------------------------------------
U statistic: 257.5000
p-value:     0.032705
p (BH-FDR):  0.192696
------------------------------------------------------------------------------------------------------------------------
   response_code  count      response_text    most_frequent
0            1.0     10  strongly disagree                 
1            2.0     17           disagree  ⭐ most frequent
2            3.0      9  slightly disagree                 
3            4.0      9     slightly agree                 
4            5.0      9              agree                 
5            6.0      2     strongly agree                 
6            NaN      2                NaN                 

VOICE QUESTION:
I would ask for human help only for special requests, such as menu changes or unusual needs
---

In [22]:
def print_crosstab_pretty_voice(q, U, p, p_bh, df_use, group_col="modality_pref"):
    ct = pd.crosstab(df_use[group_col], df_use[q])
    ct = ct.reindex(columns=[1, 2, 3, 4, 5, 6], fill_value=0).rename(columns=LIKERT_LABELS)

    headers = ["Group"] + list(ct.columns)
    rows = [[idx] + [int(v) for v in ct.loc[idx].values] for idx in ct.index]

    # column widths
    widths = [max(len(str(x)) for x in col) for col in zip(headers, *rows)]

    def fmt_row(r):
        return " | ".join(str(val).ljust(w) for val, w in zip(r, widths))

    print("\nVOICE QUESTION:")
    print(q)

    u_txt  = f"{U:.4f}" if pd.notna(U) else "NA"
    p_txt  = f"{p:.6f}" if pd.notna(p) else "NA"
    bh_txt = f"{p_bh:.6f}" if pd.notna(p_bh) else "NA"

    print(f"U = {u_txt} | p = {p_txt} | p_BH = {bh_txt}\n")

    print(fmt_row(headers))
    print("-" * (sum(widths) + 3 * (len(widths) - 1)))
    for r in rows:
        print(fmt_row(r))


In [23]:
res_sorted = res.sort_values(["p_fdr_bh", "p_value"], na_position="last")

for _, row in res_sorted.iterrows():
    q = row["voice_item"]
    print_crosstab_pretty_voice(
        q=q,
        U=row["U_stat"],
        p=row["p_value"],
        p_bh=row["p_fdr_bh"],
        df_use=df_hd,
        group_col="modality_pref"
    )



VOICE QUESTION:
Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming
U = 257.5000 | p = 0.032705 | p_BH = 0.192696

Group   | strongly disagree | disagree | slightly disagree | slightly agree | agree | strongly agree
----------------------------------------------------------------------------------------------------
digital | 3                 | 6        | 3                 | 3              | 7     | 2             
human   | 7                 | 11       | 6                 | 6              | 2     | 0             

VOICE QUESTION:
I would ask for human help only for special requests, such as menu changes or unusual needs
U = 279.5000 | p = 0.076131 | p_BH = 0.192696

Group   | strongly disagree | disagree | slightly disagree | slightly agree | agree | strongly agree
----------------------------------------------------------------------------------------------------
digital | 1                 | 0        | 6                 | 2              | 9     | 6    

In [58]:
import numpy as np
import pandas as pd

def merge_age_3grp(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54"]:
        return "35–54"
    elif age in ["55–64", "65+"]:
        return "55+"
    else:
        return np.nan

df_voice = df.copy()
df_voice["Age_3grp"] = df_voice["Age band"].apply(merge_age_3grp)


In [59]:
df_voice["Age_3grp"].value_counts(dropna=False)


Age_3grp
18–34    36
35–54    17
55+       5
NaN       4
Name: count, dtype: int64

In [60]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

results_kw = []

for q in voice_cols:
    groups = [
        df_voice.loc[df_voice["Age_3grp"] == "18–34", q].dropna(),
        df_voice.loc[df_voice["Age_3grp"] == "35–54", q].dropna(),
        df_voice.loc[df_voice["Age_3grp"] == "55+", q].dropna()
    ]

    # Check group sizes explicitly
    sizes = [len(g) for g in groups]

    if min(sizes) < 3:
        H, p = np.nan, np.nan
    else:
        H, p = kruskal(*groups)

    results_kw.append({
        "voice_item": q,
        "n_18_34": sizes[0],
        "n_35_54": sizes[1],
        "n_55_plus": sizes[2],
        "H_stat": H,
        "p_value": p
    })

res_kw = pd.DataFrame(results_kw)
res_kw


,voice_item,n_18_34,n_35_54,n_55_plus,H_stat,p_value
0,I would feel comfortable speaking to a voice-based system in a restaurant,36,17,5,0.396644,0.820106
1,Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming,35,17,4,0.587814,0.745346
2,"Browsing or selecting menu items by speaking would feel easy and natural, even with voice feedback",36,17,4,0.782475,0.676219
3,I would trust a voice-based system to correctly understand what I say,35,17,5,0.261823,0.877295
4,"I would ask for human help only for special requests, such as menu changes or unusual needs",36,15,5,3.780451,0.151038
5,I would feel confident making payments through a voice-based system,35,17,5,2.511255,0.284897
6,I would be willing to use a voice-based system on my next restaurant visit,36,16,5,8.838602,0.012043


In [61]:
import numpy as np
import pandas as pd

def merge_age_2grp(age):
    if pd.isna(age):
        return np.nan
    elif age in ["18–24", "25–34"]:
        return "18–34"
    elif age in ["35–44", "45–54", "55–64", "65+"]:
        return "35+"
    else:
        return np.nan

df_voice = df.copy()
df_voice["Age_2grp"] = df_voice["Age band"].apply(merge_age_2grp)

# Sanity check
df_voice["Age_2grp"].value_counts(dropna=False)


Age_2grp
18–34    36
35+      22
NaN       4
Name: count, dtype: int64

In [62]:
for q in voice_cols:
    df_voice[q] = pd.to_numeric(df_voice[q], errors="coerce")


In [63]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

results_mw = []

for q in voice_cols:
    g1 = (
        df_voice.loc[df_voice["Age_2grp"] == "18–34", q]
        .dropna()
        .to_numpy(dtype=float)
    )
    g2 = (
        df_voice.loc[df_voice["Age_2grp"] == "35+", q]
        .dropna()
        .to_numpy(dtype=float)
    )

    n1, n2 = len(g1), len(g2)

    if n1 < 3 or n2 < 3:
        U, p = np.nan, np.nan
    else:
        U, p = mannwhitneyu(g1, g2, alternative="two-sided")

    results_mw.append({
        "voice_item": q,
        "n_18_34": n1,
        "n_35_plus": n2,
        "U_stat": U,
        "p_value": p
    })

res_mw = pd.DataFrame(results_mw)
res_mw


,voice_item,n_18_34,n_35_plus,U_stat,p_value
0,I would feel comfortable speaking to a voice-based system in a restaurant,0,0,NaN,NaN
1,Being greeted or guided by a voice-enabled robot would feel pleasant and welcoming,0,0,NaN,NaN
2,"Browsing or selecting menu items by speaking would feel easy and natural, even with voice feedback",0,0,NaN,NaN
3,I would trust a voice-based system to correctly understand what I say,0,0,NaN,NaN
4,"I would ask for human help only for special requests, such as menu changes or unusual needs",0,0,NaN,NaN
5,I would feel confident making payments through a voice-based system,0,0,NaN,NaN
6,I would be willing to use a voice-based system on my next restaurant visit,0,0,NaN,NaN


In [64]:
for c in voice_cols:
    df_voice[c] = pd.to_numeric(df_voice[c], errors="coerce")


In [65]:
if any(len(g) < 3 for g in groups):
    H, p = np.nan, np.nan


In [66]:
df_voice.groupby("Age_3grp")[voice_cols[0]].count()


KeyError: 'Age_3grp'

In [ ]:
# df_voice

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

results_kw = []

for q in voice_cols:
    groups = [
        df_voice.loc[df_voice["Age_3grp"] == g, q].dropna()
        for g in ["18–34", "35–54", "55+"]
    ]

    # Require data in all groups
    if any(len(g) < 3 for g in groups):
        H, p = np.nan, np.nan
    else:
        H, p = kruskal(*groups)

    results_kw.append({
        "voice_item": q,
        "H_stat": H,
        "p_value": p
    })

res_kw = pd.DataFrame(results_kw)

# FDR correction across 7 voice items
mask = res_kw["p_value"].notna()
res_kw["p_fdr_bh"] = np.nan

if mask.sum() > 0:
    res_kw.loc[mask, "p_fdr_bh"] = multipletests(
        res_kw.loc[mask, "p_value"],
        alpha=0.05,
        method="fdr_bh"
    )[1]

res_kw = res_kw.sort_values(
    ["p_fdr_bh", "p_value"],
    na_position="last"
).reset_index(drop=True)

res_kw


In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

pairs = [
    ("18–34", "35–54"),
    ("18–34", "55+"),
    ("35–54", "55+")
]

pairwise_rows = []
raw_pvals = []

for _, row in kw_df.iterrows():

    if not row["significant"]:
        continue

    q = row["question"]

    temp_rows = []
    temp_pvals = []

    for g1, g2 in pairs:

        x = df_human_answered.loc[
            df_human_answered[age_col] == g1, q
        ].dropna()

        y = df_human_answered.loc[
            df_human_answered[age_col] == g2, q
        ].dropna()

        if len(x) == 0 or len(y) == 0:
            continue

        U, p = mannwhitneyu(x, y, alternative="two-sided")

        temp_pvals.append(p)

        temp_rows.append({
            "question": q,
            "group1": g1,
            "group2": g2,
            "U": U,
            "n1": len(x),
            "n2": len(y),
            "median1": np.median(x),
            "median2": np.median(y),
            "p_raw": p
        })

    # ✅ Benjamini–Hochberg within THIS question
    if temp_pvals:

        reject, p_bh, _, _ = multipletests(temp_pvals, method="fdr_bh")

        for r, pbh, sig in zip(temp_rows, p_bh, reject):
            r["p_bh"] = pbh
            r["significant_bh"] = sig
            pairwise_rows.append(r)

pairwise_df = pd.DataFrame(pairwise_rows)

pairwise_df


In [ ]:
likert_cols = voice_cols

In [70]:
from scipy.stats import kruskal
import pandas as pd
import numpy as np

# age_col = "Age_3grp"
age_col = df_voice.loc[df_voice["Age_3grp"]]

alpha = 0.05
age_levels = ["18–34", "35–54", "55+"]

kw_rows = []

for q in likert_cols:

    samples = []
    present_groups = []

    for g in age_levels:
        vals = df_voice.loc[
            df_voice[age_col] == g, q
        ].dropna()

        if len(vals) > 0:
            samples.append(vals)
            present_groups.append(g)

    if len(samples) < 2:
        continue

    H, p_kw = kruskal(*samples)

    kw_rows.append({
        "question": q,
        "H": H,
        "p_kw": p_kw,
        "n_total": int(sum(len(s) for s in samples)),
        "groups_used": ", ".join(present_groups),
        "significant": p_kw < alpha
    })

kw_df = pd.DataFrame(kw_rows).sort_values("p_kw")

kw_df


KeyError: 'Age_3grp'

In [72]:
import numpy as np
import pandas as pd
from scipy.stats import kruskal

results_kw = []

age_col = "Age_3grp"
age_levels = ["18–34", "35–54", "55+"]

for q in voice_cols:

    groups = []
    sizes = []

    for g in age_levels:
        vals = df_voice.loc[df_voice[age_col] == g, q].dropna()
        groups.append(vals)
        sizes.append(len(vals))

    if min(sizes) < 3:
        H, p = np.nan, np.nan
    else:
        H, p = kruskal(*groups)

    results_kw.append({
        "voice_item": q,
        "n_18_34": sizes[0],
        "n_35_54": sizes[1],
        "n_55_plus": sizes[2],
        "H_stat": H,
        "p_value": p
    })

res_kw = pd.DataFrame(results_kw)

res_kw


KeyError: 'Age_3grp'